# Step 2: Task-Agnostic θ_BS (KILLER EXPERIMENT)

**Claim**: θ_BS trained on Task A (channel estimation) transfers to Task B (beam prediction).

## Experiment Design
1. Train on Task A (channel estimation): learn E_A + θ_task_A + θ_BS
2. Freeze θ_BS
3. Train new θ_task_B for Task B (beam prediction), using frozen E_A and θ_BS
4. Compare vs. training Task B from scratch (no θ_BS)

If θ_BS helps Task B → it captures site-specific info independent of task → "site foundation representation"

In [ ]:
import sys, os
sys.path.insert(0, '/workspace')
os.chdir('/workspace')

import copy
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

from src.data.dataset import ChannelEstimationDataset
from src.models.estimator import (
    SiteAwareEstimator, SharedEncoder, TaskHead, 
    SiteEmbedding, FiLMSiteInjection, create_model
)
from src.training.trainer import train_local, evaluate

device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
DATA_DIR = 'data/channels'
TARGET_BS = 6
BATCH_SIZE = 32

## Phase 1: Train Task A (Channel Estimation) → learn θ_BS

In [ ]:
# Train channel estimation model with θ_BS on target BS
ds_a = ChannelEstimationDataset(data_dir=DATA_DIR, bs_ids=[TARGET_BS])
n = len(ds_a)
n_val = int(n * 0.2)
train_a, val_a = torch.utils.data.random_split(ds_a, [n - n_val, n_val])

model_a = create_model(site_integration='film', site_embed_dim=64).to(device)
res_a = train_local(
    model_a, DataLoader(train_a, batch_size=BATCH_SIZE, shuffle=True),
    DataLoader(val_a, batch_size=BATCH_SIZE), epochs=100, device=device,
)
print(f'Task A (CE) NMSE: {10*np.log10(res_a["best_val"]):.2f} dB')
print(f'Learned θ_BS norm: {model_a.site_embedding.embedding.norm():.4f}')

## Phase 2: Transfer θ_BS to Task B

For Task B, we simulate a simplified beam prediction task:
- Input: same channel H
- Target: optimal beam index (argmax of beamforming gain)

*Note: Full beam prediction requires codebook design. Here we use a simplified version 
where Task B = predict the dominant eigenvector direction of the channel matrix.*

In [ ]:
# Task B: Predict channel power profile (simplified beam-related task)
# Uses same encoder, different task head
# Target: |H|^2 power across subcarriers (power delay profile analog)

class TaskBHead(nn.Module):
    """Task B head: predict channel power profile."""
    def __init__(self, in_channels=64, n_output=1024):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d((1, n_output))
        self.fc = nn.Sequential(
            nn.Linear(in_channels, 128),
            nn.ReLU(),
            nn.Linear(128, 1),
        )
    
    def forward(self, feat):
        # feat: (B, C, H, W) → pool to (B, C, 1, W) → permute
        B, C, H, W = feat.shape
        x = feat.mean(dim=2)  # (B, C, W) - avg over antenna dim
        x = x.permute(0, 2, 1)  # (B, W, C)
        x = self.fc(x).squeeze(-1)  # (B, W)
        return x


class TransferModel(nn.Module):
    """Model for Task B using frozen E and θ_BS from Task A."""
    def __init__(self, encoder, site_embedding, site_injection, task_b_head):
        super().__init__()
        self.encoder = encoder
        self.site_embedding = site_embedding
        self.site_injection = site_injection
        self.task_b_head = task_b_head
    
    def forward(self, x):
        feat = self.encoder(x)
        if self.site_injection is not None:
            feat = self.site_injection(feat, self.site_embedding.embedding)
        return self.task_b_head(feat)

In [ ]:
# --- Method A: Transfer θ_BS (freeze E + θ_BS, train θ_task_B only) ---
transfer_model = TransferModel(
    encoder=copy.deepcopy(model_a.encoder),
    site_embedding=copy.deepcopy(model_a.site_embedding),
    site_injection=copy.deepcopy(model_a.site_injection),
    task_b_head=TaskBHead(64, 1024),
).to(device)

# Freeze encoder and site embedding
for p in transfer_model.encoder.parameters():
    p.requires_grad = False
for p in transfer_model.site_injection.parameters():
    p.requires_grad = False
transfer_model.site_embedding.embedding.requires_grad = False

print('Transfer model trainable params:', 
      sum(p.numel() for p in transfer_model.parameters() if p.requires_grad))

# --- Method B: No θ_BS (only E, train θ_task_B) ---
no_site_model = TransferModel(
    encoder=copy.deepcopy(model_a.encoder),
    site_embedding=SiteEmbedding(64),  # fresh, zero
    site_injection=None,  # disabled
    task_b_head=TaskBHead(64, 1024),
).to(device)
for p in no_site_model.encoder.parameters():
    p.requires_grad = False

# --- Method C: From scratch for Task B ---
scratch_model = TransferModel(
    encoder=SharedEncoder(2, 64, 3, 3),
    site_embedding=SiteEmbedding(64),
    site_injection=None,
    task_b_head=TaskBHead(64, 1024),
).to(device)

In [ ]:
# Task B data: same input, target = power profile
# We create a custom collate that computes power profile as target

def task_b_collate(batch):
    inputs = torch.stack([b['input'] for b in batch])
    targets = torch.stack([b['target'] for b in batch])
    # Power profile: mean |H|^2 across antenna pairs
    power = (targets ** 2).sum(dim=1).mean(dim=1)  # (B, n_sc)
    return {'input': inputs, 'target': power}

train_b_loader = DataLoader(train_a, batch_size=BATCH_SIZE, shuffle=True, collate_fn=task_b_collate)
val_b_loader = DataLoader(val_a, batch_size=BATCH_SIZE, collate_fn=task_b_collate)

print('Task B target shape check:')
sample = next(iter(train_b_loader))
print(f'  Input: {sample["input"].shape}, Target: {sample["target"].shape}')

In [ ]:
# Train Task B models
# Custom training for regression task (MSE loss instead of NMSE)

def train_task_b(model, train_loader, val_loader, epochs=50, lr=1e-3):
    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()), lr=lr
    )
    criterion = nn.MSELoss()
    train_losses, val_losses = [], []
    
    for epoch in range(epochs):
        model.train()
        total = 0
        n = 0
        for batch in train_loader:
            x = batch['input'].to(device)
            y = batch['target'].to(device)
            optimizer.zero_grad()
            pred = model(x)
            loss = criterion(pred, y)
            loss.backward()
            optimizer.step()
            total += loss.item() * x.size(0)
            n += x.size(0)
        train_losses.append(total / n)
        
        model.eval()
        total = 0
        n = 0
        with torch.no_grad():
            for batch in val_loader:
                x = batch['input'].to(device)
                y = batch['target'].to(device)
                pred = model(x)
                total += criterion(pred, y).item() * x.size(0)
                n += x.size(0)
        val_losses.append(total / n)
        
        if (epoch + 1) % 10 == 0:
            print(f'  Epoch {epoch+1}: train={train_losses[-1]:.6f}, val={val_losses[-1]:.6f}')
    
    return {'train': train_losses, 'val': val_losses}

print('Training with θ_BS transfer...')
res_transfer = train_task_b(transfer_model, train_b_loader, val_b_loader)

print('\nTraining without θ_BS...')
res_no_site = train_task_b(no_site_model, train_b_loader, val_b_loader)

print('\nTraining from scratch...')
res_scratch = train_task_b(scratch_model, train_b_loader, val_b_loader)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(res_transfer['val'], label='E + θ_BS transfer (ours)', linewidth=2)
ax.plot(res_no_site['val'], label='E transfer only (no θ_BS)', linewidth=2)
ax.plot(res_scratch['val'], label='From scratch', linewidth=2)

ax.set_xlabel('Epoch')
ax.set_ylabel('MSE')
ax.set_title(f'Task B (Power Profile) with Transferred θ_BS from Task A - BS{TARGET_BS}')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('step2_task_agnostic.png', dpi=150)
plt.show()

print(f'\nFinal val MSE:')
print(f'  With θ_BS:    {res_transfer["val"][-1]:.6f}')
print(f'  Without θ_BS: {res_no_site["val"][-1]:.6f}')
print(f'  From scratch: {res_scratch["val"][-1]:.6f}')
print(f'  Improvement:  {(res_no_site["val"][-1] - res_transfer["val"][-1]) / res_no_site["val"][-1] * 100:.1f}%')